<a href="https://colab.research.google.com/github/NikitaMarnykh/labor_market_analysis/blob/main/%D0%9C%D0%B0%D1%80%D0%BD%D1%8B%D1%85_%D0%9D_%D0%92_%D0%B8_%D0%9E%D0%B1%D0%BE%D1%80%D0%BE%D1%82%D0%BE%D0%B2_%D0%9C_%D0%92_%D1%80%D1%8B%D0%BD%D0%BE%D0%BA_%D1%82%D1%80%D1%83%D0%B4%D0%B0_%D1%80%D0%B5%D0%B3%D1%80%D0%B5%D1%81%D1%81%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_%D0%B8_%D0%BF%D0%BE%D1%81%D1%82%D1%80%D0%BE%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B5%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Признаки, которые будут использованы для дальнейшего анализа**

Количественные признаки:
  
  - идентификатор вакансии (id)

  - минимальная заработная плат (salary_from)
  
  - максимальная заработная плата (salary_to)

Категориальные признаки:

  - профессиональная роль (professional roles)

  - ключевые навыки (key_skills)

  - наименование работодателя (employer)

  - опыт работы (experience)

  - график работы (schedule)

  - тип занятости (employment)

  - доступна ли вакансия для соискателей с инвалидностью (accept_handicapped)

  - город (city)

  - сфера деятельности (field_of_activity)

## 19. Установка и подключение зависимостей для регрессионного анализа и построения моделей

Устанавливаем все необходимые зависимости

In [556]:
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install statsmodels

Подключаем все необходимые зависимости

In [557]:
from sklearn import linear_model as lm
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.stats.diagnostic import normal_ad
from statsmodels.stats.outliers_influence import variance_inflation_factor

import numpy as np

import pandas as pd

Подгружаем репозиторий для работы с ним

In [558]:
!git clone https://github.com/NikitaMarnykh/labor_market_analysis

Cloning into 'labor_market_analysis'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 118 (delta 73), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 69.98 MiB | 7.56 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Updating files: 100% (11/11), done.


Перейдём в установленный репозиторий

In [559]:
%cd labor_market_analysis/

/content/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis/labor_market_analysis


Сохраним датасет в переменную

In [560]:
dataset = pd.read_csv('cleaned_hh_hard.csv',
                      na_values=['NA', 'N/A', 'null', 'missing', '-', '?', '...'],
                      dtype={
                             'id': 'int64', 'professional_roles': 'string',
                             'experience': 'category', 'schedule': 'category',
                             'employment': 'category', 'employer': 'string',
                             'accept_handicapped': 'category', 'key_skills': 'string',
                             'city': 'string', 'salary_from': 'int64',
                             'salary_to': 'int64', 'field_of_activity': 'category',

                      },
                      keep_default_na=True,
                      na_filter=True)

Посмотрим информацию о датасете

In [561]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81412 entries, 0 to 81411
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   id                  81412 non-null  int64   
 1   professional_roles  81412 non-null  string  
 2   experience          81412 non-null  category
 3   schedule            81412 non-null  category
 4   employment          81412 non-null  category
 5   employer            81412 non-null  string  
 6   accept_handicapped  81412 non-null  category
 7   key_skills          81412 non-null  string  
 8   city                81412 non-null  string  
 9   salary_from         81412 non-null  int64   
 10  salary_to           81412 non-null  int64   
 11  field_of_activity   81412 non-null  category
dtypes: category(5), int64(3), string(4)
memory usage: 4.7 MB


## 20. Регрессионный анализ. Построение моделей.



Создадим DataFrame для хранения метрик

In [562]:
quality_metrics = [
    'R2_train', 'R2_test',
    'Adj_R2_train', 'Adj_R2_test',
    'MSE_train', 'MSE_test',
    'RMSE_train', 'RMSE_test',
    'MAE_train', 'MAE_test',
    'Train_Jarque_Bera_statistic', 'Test_Jarque_Bera_statistic',
    'Train_Jarque_Bera_pvalue', 'Test_Jarque_Bera_pvalue',
    'Train_skewness', 'Test_skewness',
    'Train_kurtosis', 'Test_kurtosis',
    'Train_Durbin_Watson', 'Test_Durbin_Watson',
    'Train_Anderson_Darling_statistics', 'Test_Anderson_Darling_statistics',
    'Train_Anderson_Darling_pvalue', 'Test_Anderson_Darling_pvalue'
]
models = pd.DataFrame(index=quality_metrics)

Напишем формулу для вычисления скорректированного коэффициента детерминации

In [563]:
def calculate_adjusted_coefficient_of_determination(r2_score: float,
                                                    number_of_observations: int,
                                                    number_of_features: int) -> float:
    """Функция вычисляющая скорректированый коэффициент детерминации"""

    return 1 - (1 - r2_score) * (number_of_observations - 1) / (number_of_observations - number_of_features - 1)

### 20.1 Labeb Encoder

Создадим label-энкодер

In [564]:
label_encoder = LabelEncoder()

Список столбцов для кодирования

In [565]:
columns = ['professional_roles', 'employer', 'key_skills', 'city', 'field_of_activity']

Обучим его и трансформируем данные

In [566]:
for column in columns:
    dataset[column] = label_encoder.fit_transform(dataset[column])

### 20.2 One Hot Encoder

Создадим One-hot-энкодер

In [567]:
one_hot_encoder = OneHotEncoder(
                                categories='auto',                      # Определяет категории для каждого признака
                                drop='first',                           # Позволяет удалить одну из колонок, чтобы избежать мультиколлинеарности
                                sparse_output=False,                    # Определяет, возввращать ли результат в формате разреженной матрицы
                                dtype=np.int8,                          # Определяет тип данных выходного массива
                                handle_unknown='error',                 # Определяет, как обрабатывать категории, которые не встречались при обучении
                                min_frequency=None,                     # Определяет минимальную частоту категории, чтобы она была включена в кодировку
                                max_categories=None,                    # Определяет максимальное количество категорий для каждого признака
                                feature_name_combiner="concat",         # Параметр управляет формированием названий признаков
                                )

Список столбцов для кодирования

In [568]:
columns = ['schedule', 'employment', 'accept_handicapped']

Обучим его и трансформируем данные

In [569]:
encoded_data = one_hot_encoder.fit_transform(dataset[columns])

Получение имён новых столбцов

In [570]:
new_feature_names = one_hot_encoder.get_feature_names_out(columns)

Cоздание DataFrame из закодированных данных

In [571]:
encoded_dataframe = pd.DataFrame(encoded_data, columns=new_feature_names)

Удаляем старые категориальные столбцы и добавляем закодированные

In [572]:
dataset = pd.concat([dataset.drop(columns=columns), encoded_dataframe], axis=1)

### 20.3 Ordinal Encoder

Создадим ordinal-энкодер

In [573]:
one_hot_encoder = OrdinalEncoder(
                                categories='auto',            # Определяет категории для каждого признака
                                dtype=np.int8,                # Определяет тип данных выходного массива
                                handle_unknown='error',       # Определяет, как обрабатывать категории, которые не встречались при обучении
                                unknown_value=None,           # Определяет значение для замены неизвестных категорий
                                encoded_missing_value=-1,     # Определяет число для замены пропущенных значений
                                min_frequency=None,           # Определяет минимальную частоту категории, чтобы она была включена в кодировку
                                max_categories=None,          # Определяет максимальное количество категорий для каждого признака
                                )

Список столбцов для кодирования

In [574]:
columns = ['experience']

Обучим его и трансформируем данные

In [575]:
dataset[columns] = one_hot_encoder.fit_transform(dataset[columns])

### 20.4 Результат

Посмотрим информацию о датасете

In [576]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81412 entries, 0 to 81411
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    81412 non-null  int64
 1   professional_roles    81412 non-null  int64
 2   experience            81412 non-null  int8 
 3   employer              81412 non-null  int64
 4   key_skills            81412 non-null  int64
 5   city                  81412 non-null  int64
 6   salary_from           81412 non-null  int64
 7   salary_to             81412 non-null  int64
 8   field_of_activity     81412 non-null  int64
 9   schedule_full_day     81412 non-null  int8 
 10  schedule_labor        81412 non-null  int8 
 11  schedule_remote       81412 non-null  int8 
 12  schedule_shift        81412 non-null  int8 
 13  employment_part       81412 non-null  int8 
 14  employment_probation  81412 non-null  int8 
 15  employment_project    81412 non-null  int8 
 16  acce

Осмотрим датасет

In [577]:
dataset.head()

,id,professional_roles,experience,employer,key_skills,city,salary_from,salary_to,field_of_activity,schedule_full_day,schedule_labor,schedule_remote,schedule_shift,employment_part,employment_probation,employment_project,accept_handicapped_1
0,72121675,54,1,3475,26,40,75000,100000,4,1,0,0,0,0,0,0,0
1,72121675,54,1,3475,52,40,75000,100000,4,1,0,0,0,0,0,0,0
2,72121675,54,1,3475,56,40,75000,100000,4,1,0,0,0,0,0,0,0
3,72121675,54,1,3475,204,40,75000,100000,4,1,0,0,0,0,0,0,0
4,72121675,54,1,3475,680,40,75000,100000,4,1,0,0,0,0,0,0,0


### 20.5 Подготовка выборок

В рамках подготовки данных к моделированию осуществим процедуру разделения исходного набора наблюдений. В качестве зависимых переменных выделим показатели salary_from и salary_to, тогда как остальные параметры рассматриваются как независимые факторы (предикторы).

In [578]:
target_variables = dataset[['salary_from', 'salary_to']]
factor_variables = dataset.drop(['salary_from', 'salary_to', 'id', 'city'], axis=1)

Для последующей валидации качества модели реализуем стратифицированное разбиение выборки на обучающее и тестовое подмножества в соотношении 80:20.

In [579]:
factors_train, factors_test, targets_train, targets_test = train_test_split(factor_variables, target_variables, test_size = 0.2, random_state = 0)

Проверим мультиколлинеарность признаков с помощью Variance Inflation Factor (VIF)

In [580]:
vifs = [variance_inflation_factor(factor_variables.values, i)
        for i in range(factor_variables.shape[1])]

vif_df = pd.DataFrame({
    'Признак': factor_variables.columns,
    'VIF': vifs
})

vif_df

,Признак,VIF
0,professional_roles,3.932717
1,experience,2.605274
2,employer,4.219176
3,key_skills,5.771164
4,field_of_activity,4.014006
5,schedule_full_day,8.771140
6,schedule_labor,1.912814
7,schedule_remote,1.382752
8,schedule_shift,2.060326
9,employment_part,1.129056


### 20.6 Линейная регрессия

Создадим линейную регрессионную модель

In [581]:
model = MultiOutputRegressor(
                              estimator=lm.LinearRegression(                    # Базовый регрессор, который будет применяться для каждого целевого признака
                                                            fit_intercept=True, # Нужно ли вычислять свободный член в линейном уравнении
                                                            copy_X=True,        # Нужно ли копировать входные данные перед обучением
                                                            n_jobs=None,        # Количество ядер CPU для параллельных вычислений
                                                            positive=False      # Определяет будут ли коэффициенты модели неотрицательными
                                                            ),
                              n_jobs=None                                       # Количество ядер CPU для параллельных вычислений
                                          )

Обучим модель

In [582]:
model.fit(
          X=factors_train,            # Это признаки, на которых будет обучаться модель
          y=targets_train,            # Это целевые переменные
          sample_weight=None,         # Позволяет задать веса для каждого объекта в обучающей выборке
          )

MultiOutputRegressor(estimator=LinearRegression())

Оценим свободные члены моделей

In [583]:
print(f"Свободный член для первой модели:\n{model.estimators_[0].intercept_}")
print(f"Свободный член для второй модели:\n{model.estimators_[1].intercept_}")

Свободный член для первой модели:
51692.21535516162
Свободный член для второй модели:
84771.11645103882


Оценим коэффициенты моделей

In [584]:
print(f"Коэффициенты первой модели:\n{model.estimators_[0].coef_}")
print(f"Коэффициенты второй модели:\n{model.estimators_[1].coef_}")

Коэффициенты первой модели:
[-9.22348146e+00  1.11021268e+04 -9.90063629e-01  2.67373425e-01
 -7.95015368e+02 -3.92600735e+03  3.76441720e+04  3.66676053e+03
 -1.00983837e+04 -1.00076071e+04  6.11776677e+03 -2.39658391e+03
 -2.11206075e+03]
Коэффициенты второй модели:
[-3.81127435e+00  1.37910183e+04 -1.93753682e+00  8.45961715e-01
 -1.21886191e+03 -1.01919750e+04  3.53295733e+04  1.18762447e+04
 -2.32193685e+04 -9.18793676e+03  9.45506433e+03 -5.01504904e+03
  8.73321875e+03]


Займёмся прогнозированием

In [585]:
target_train_pred = model.predict(factors_train)
target_test_pred = model.predict(factors_test)

Оценим коэффициенты детерменации

In [586]:
coefficient_of_determination_train = r2_score(targets_train, target_train_pred, multioutput='raw_values')
coefficient_of_determination_test = r2_score(targets_test, target_test_pred, multioutput='raw_values')
print(f"Train R2 score: {coefficient_of_determination_train}")
print(f"Test R2 score: {coefficient_of_determination_test}")

Train R2 score: [0.33754197 0.21838981]
Test R2 score: [0.34532904 0.20862588]


Оценим скорректированные коэффициенты детерменации

In [587]:
adjusted_coefficient_of_determination_train = calculate_adjusted_coefficient_of_determination(coefficient_of_determination_train, len(targets_train), factor_variables.shape[1])
adjusted_coefficient_of_determination_test = calculate_adjusted_coefficient_of_determination(coefficient_of_determination_test, len(targets_test), factor_variables.shape[1])
print(f"Adjusted train R2 score: {adjusted_coefficient_of_determination_train}")
print(f"Adjusted test R2 score: {adjusted_coefficient_of_determination_test}")

Adjusted train R2 score: [0.33740971 0.21823376]
Adjusted test R2 score: [0.34480591 0.20799352]


Оценим среднеквадратическую ошибку

In [588]:
mean_squared_error_train = mean_squared_error(targets_train, target_train_pred, multioutput='raw_values')
mean_squared_error_test = mean_squared_error(targets_test, target_test_pred, multioutput='raw_values')
print(f"Train MSE: {mean_squared_error_train}")
print(f"Test MSE: {mean_squared_error_test}")

Train MSE: [5.35752022e+08 1.52427674e+09]
Test MSE: [5.17697981e+08 1.51462418e+09]


Оценим корень из среднеквадратической ошибки

In [589]:
root_of_the_mean_square_error_train = np.power(mean_squared_error(targets_train, target_train_pred, multioutput='raw_values'), 0.5)
root_of_the_mean_square_error_test = np.power(mean_squared_error(targets_test, target_test_pred, multioutput='raw_values'), 0.5)
print(f"Train RMSE: {root_of_the_mean_square_error_train}")
print(f"Test RMSE: {root_of_the_mean_square_error_test}")

Train RMSE: [23146.31768322 39041.98682579]
Test RMSE: [22752.9774034  38918.17290293]


Оценим среднеюю абсолютную ошибку

In [590]:
mean_absolute_error_train = mean_absolute_error(targets_train, target_train_pred, multioutput='raw_values')
mean_absolute_error_test = mean_absolute_error(targets_test, target_test_pred, multioutput='raw_values')
print(f"Train MAE: {mean_absolute_error_train}")
print(f"Test MAE: {mean_absolute_error_test}")

Train MAE: [15678.07633539 26868.69503699]
Test MAE: [15685.02014284 26985.44418711]


Оценим нормальность распределения остатков по критерию Жака-Бера, а так же по эксцессу и асимметрии

In [591]:
jarque_bera_train = jarque_bera(target_train_pred)
jarque_bera_test = jarque_bera(target_test_pred)
skewness_train = jarque_bera_train[2]
kurtosis_train = jarque_bera_train[3]
skewness_test = jarque_bera_test[2]
kurtosis_test = jarque_bera_test[3]
print(f"Train Jarque-Bera statistics: {jarque_bera_train[0]}")
print(f"Test Jarque-Bera statistics: {jarque_bera_test[0]}")
print(f"Train Jarque-Bera: {jarque_bera_train[1]}")
print(f"Test Jarque-Bera: {jarque_bera_test[1]}")
print(f"Train skewness: {skewness_train}")
print(f"Test skewness: {skewness_test}")
print(f"Train kurtosis: {kurtosis_train}")
print(f"Test kurtosis: {kurtosis_test}")

Train Jarque-Bera statistics: [32431.64590662 12377.32550553]
Test Jarque-Bera statistics: [8345.92527104 3189.03312693]
Train Jarque-Bera: [0. 0.]
Test Jarque-Bera: [0. 0.]
Train skewness: [1.35082469 0.93626456]
Test skewness: [1.36733401 0.9450226 ]
Train kurtosis: [5.15688061 4.02697234]
Test kurtosis: [5.19611005 4.06213951]


Оценим нормальность распределения остатков по критерию Андерсона-Дарлинга

In [592]:
anderson_darling_train = normal_ad(target_train_pred)
anderson_darling_test = normal_ad(target_test_pred)
print(f"Train Anderson-Darling statistics: {anderson_darling_train[0]}")
print(f"Test Anderson-Darling statistics: {anderson_darling_test[0]}")
print(f"Train Anderson-Darling: {anderson_darling_train[1]}")
print(f"Test Anderson-Darling: {anderson_darling_test[1]}")

Train Anderson-Darling statistics: [1815.92723862  939.93552329]
Test Anderson-Darling statistics: [468.38823473 237.9236386 ]
Train Anderson-Darling: [inf inf]
Test Anderson-Darling: [            inf 8.74374307e-133]


/usr/local/lib/python3.11/dist-packages/statsmodels/stats/_adnorm.py:131: RuntimeWarning: overflow encountered in exp
  pval4 = lambda ad2a: np.exp(1.2937 - 5.709 * ad2a + 0.0186 * ad2a ** 2)
/usr/local/lib/python3.11/dist-packages/statsmodels/stats/_adnorm.py:131: RuntimeWarning: overflow encountered in exp
  pval4 = lambda ad2a: np.exp(1.2937 - 5.709 * ad2a + 0.0186 * ad2a ** 2)


Оценим автокорреляцию остатков по критерию Дарбина-Уотсона

In [593]:
durbin_watson_train = durbin_watson(target_train_pred)
durbin_watson_test = durbin_watson(target_test_pred)
print(f"Train Durbin-Watson: {durbin_watson_train}")
print(f"Test Durbin-Watson: {durbin_watson_test}")

Train Durbin-Watson: [0.20081602 0.14531659]
Test Durbin-Watson: [0.20088269 0.14472324]


Сохраним метрики в список

In [594]:
metrics_values = [
    coefficient_of_determination_train,
    coefficient_of_determination_test,
    adjusted_coefficient_of_determination_train,
    adjusted_coefficient_of_determination_test,
    mean_squared_error_train,
    mean_squared_error_test,
    root_of_the_mean_square_error_train,
    root_of_the_mean_square_error_test,
    mean_absolute_error_train,
    mean_absolute_error_test,
    jarque_bera_train[0],
    jarque_bera_test[0],
    jarque_bera_train[1],
    jarque_bera_test[1],
    skewness_train,
    skewness_test,
    kurtosis_train,
    kurtosis_test,
    durbin_watson_train,
    durbin_watson_test,
    anderson_darling_train[0],
    anderson_darling_test[0],
    anderson_darling_train[1],
    anderson_darling_test[1]
]

Заполним DataFrame с метриками модели

In [595]:
models[model.estimator.__class__.__name__] = metrics_values
models

,LinearRegression
R2_train,"[0.33754196761581234, 0.21838980692007104]"
R2_test,"[0.345329036134733, 0.2086258846984339]"
Adj_R2_train,"[0.33740971000357256, 0.21823376096276415]"
Adj_R2_test,"[0.3448059110176238, 0.20799352478086552]"
MSE_train,"[535752022.29252326, 1524276735.3047967]"
MSE_test,"[517697980.7196372, 1514624182.1023512]"
RMSE_train,"[23146.317683219575, 39041.986825785345]"
RMSE_test,"[22752.977403400135, 38918.17290292995]"
MAE_train,"[15678.076335385183, 26868.69503699211]"
MAE_test,"[15685.02014283698, 26985.444187105422]"
